# Beef Tenderness Data Merge
## Step-by-Step Guide to Combine Sensory Panel and Carcass Data

This notebook will:
1. Check if each carcass has all 4 primal cuts in the sensory data
2. Identify any missing cuts or carcasses
3. Merge the sensory panel data with carcass characteristics
4. Create both combined and separate datasets for modeling

In [ ]:
"""
=============================================================
 NOTEBOOK 1 — Data Merge & Dataset Construction
=============================================================
This notebook constructs the modelling dataset by joining two 
separate data sources:

  - Sensory Panel Data: individual panellist eating quality 
    scores per animal per cut (Tenderness, Juiciness, Flavour, 
    Overall Liking)
  - Carcass Data: physical grading measurements, breed, kill 
    date, fat cover, conformation, and chill curve time-series

A key design decision is made here: rather than averaging panel 
scores to one row per animal, each individual panellist evaluation 
is retained as a separate row. This expands the dataset from 
~400 carcass-level records to ~10,900 panellist-level records, 
exposing the true within-animal disagreement that averaging 
would mask.

Outputs:
  - Combined panellist-level dataset per primal cut
  - Carcass-level dataset for feature engineering
=============================================================
"""

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pandas.plotting import scatter_matrix
import seaborn as sns
import scipy.stats as stats
import statsmodels.api as sm
import csv
from scipy.stats import chi2_contingency
import matplotlib.pyplot as plt

from openpyxl import Workbook
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.styles import Font, Alignment
import re
import textwrap
from statsmodels.stats.contingency_tables import mcnemar
import sqlite3
from contextlib import contextmanager
import json
import csv
from fuzzywuzzy import process
from itertools import product

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)



In [ ]:
def convert_columns_to_float(df, columns):
    """
    Converts specified columns in a DataFrame to float.

    Parameters:
    - df (pd.DataFrame): The DataFrame containing the columns to be converted.
    - columns (list of str): List of column names to convert to float.

    Returns:
    - pd.DataFrame: DataFrame with specified columns converted to float.
    """
    for column in columns:
        if column in df.columns:
            try:
                df[column] = df[column].astype(str).str.replace(r'[^\d.]', '', regex=True) # fix hidden chars
                df[column] = pd.to_numeric(df[column], errors='coerce') # fix hidden chars
                df[column] = df[column].astype("int64")
                print(f"Column {column} converted to float successfully.")
            except ValueError:
                print(f"Column {column} could not be converted to float due to non-numeric values.")
        else:
            print(f"Column {column} not found in DataFrame.")
    return df


## 1. Load the Data

**TODO: Update file paths to your actual files**

In [ ]:
# Load sensory panel data
path =  "C:"
###### use this script to test panel v carcass    and measure v carcas
sensory_panel_data = f"{path}Panel Sensory Data_fixed_2.2.xlsx"
panel_df = pd.read_excel(sensory_panel_data)  # UPDATE THIS PATH

carcass_data = f"{path}carcass_8.7.1.xlsx"
# Load carcass data
carcass_df = pd.read_excel(carcass_data)  # UPDATE THIS PATH

# load measure data
measure_data = f"{path}Measure_DATA_CleanedUp_2.3.xlsx"
measure_df = pd.read_excel(measure_data)  # UPDATE THIS PATH

print(f"Sensory panel data shape: {panel_df.shape}")
print(f"Carcass data shape: {carcass_df.shape}")
print(f"\nSensory panel columns: {list(panel_df.columns)}")
print(f"\nFirst few sensory records:")
panel_df.head()

In [ ]:
floats_features  = ['Eartag']
panel_df = convert_columns_to_float(panel_df, floats_features)
#panel_df = convert_columns_to_float(panel_df, floats_features)

In [ ]:
panel_df.info(max_cols = 300, show_counts = True, memory_usage="deep")

In [ ]:
carcass_df.info(max_cols = 300, show_counts = True, memory_usage="deep")

## 2. Check Data Completeness

### 2.1 Check Primal Codes in Sensory Data

In [ ]:
# What primal codes do we have?
print("Unique primal codes in sensory data:")
print(panel_df['Primal'].value_counts())

# Expected primal codes
expected_primals = ['LD', 'SM', 'GM', 'FT']
actual_primals = panel_df['Primal'].unique()

print(f"\nExpected primals: {expected_primals}")
print(f"Actual primals: {list(actual_primals)}")
print(f"Missing primals: {set(expected_primals) - set(actual_primals)}")
print(f"Unexpected primals: {set(actual_primals) - set(expected_primals)}")

### 2.2 Check Which Carcasses Have All 4 Cuts

In [ ]:
# Count how many cuts each carcass has
cuts_per_carcass = panel_df.groupby('Eartag')['Primal'].nunique().reset_index()
cuts_per_carcass.columns = ['Eartag', 'Number_of_Cuts']

print("Distribution of cuts per carcass:")
print(cuts_per_carcass['Number_of_Cuts'].value_counts().sort_index())

# Summary statistics
print(f"\nTotal unique carcasses: {cuts_per_carcass.shape[0]}")
print(f"Carcasses with all 4 cuts: {(cuts_per_carcass['Number_of_Cuts'] == 4).sum()}")
print(f"Carcasses with < 4 cuts: {(cuts_per_carcass['Number_of_Cuts'] < 4).sum()}")

# Show carcasses with incomplete data
incomplete = cuts_per_carcass[cuts_per_carcass['Number_of_Cuts'] < 4]
if len(incomplete) > 0:
    print(f"\n⚠️ WARNING: {len(incomplete)} carcasses have incomplete cut data")
    print("\n carcasses with missing cuts:")
    print(incomplete.head(50))
    
    # Show which cuts are missing for these carcasses
    print("\nMissing cuts detail (first 5 incomplete carcasses):")
    for eartag in incomplete['Eartag'].head(50):
        available_cuts = panel_df[panel_df['Eartag'] == eartag]['Primal'].unique()
        missing_cuts = set(expected_primals) - set(available_cuts)
        print(f"  Eartag {eartag}: has {list(available_cuts)}, missing {list(missing_cuts)}")
else:
    print("\n✅ All carcasses have all 4 cuts!")

### 2.3 Check Panelist Coverage

In [ ]:
# Check how many panelists evaluated each cut
panelists_per_cut = panel_df.groupby(['Eartag', 'Primal'])['Username'].nunique().reset_index()
panelists_per_cut.columns = ['Eartag', 'Primal', 'Number_of_Panelists']

print("Distribution of panelists per cut:")
print(panelists_per_cut['Number_of_Panelists'].value_counts().sort_index())

# Check for cuts with < 6 panelists
incomplete_panels = panelists_per_cut[panelists_per_cut['Number_of_Panelists'] < 6]
if len(incomplete_panels) > 0:
    print(f"\n⚠️ WARNING: {len(incomplete_panels)} cuts have < 6 panelists")
    print("\nFirst 10 cuts with incomplete panels:")
    print(incomplete_panels.head(60))
else:
    print("\n✅ All cuts have 6 panelists!")

# Overall panelist info
print(f"\nTotal unique panelists: {panel_df['Username'].nunique()}")
print("\nPanelist contribution:")
print(panel_df['Username'].value_counts())

### 2.4 Check for Eartag Matches Between Datasets

In [ ]:
# Get unique eartags from both datasets
panel_eartags = set(panel_df['Eartag'].unique())
carcass_eartags = set(carcass_df['Eartag'].unique())

print(f"Eartags in sensory panel data: {len(panel_eartags)}")
print(f"Eartags in carcass data: {len(carcass_eartags)}")

# Check overlap
common_eartags = panel_eartags & carcass_eartags
panel_only = panel_eartags - carcass_eartags
carcass_only = carcass_eartags - panel_eartags

print(f"\nEartags in BOTH datasets: {len(common_eartags)}")
print(f"Eartags ONLY in sensory data: {len(panel_only)}")
print(f"Eartags ONLY in carcass data: {len(carcass_only)}")

if len(panel_only) > 0:
    print(f"\n⚠️ WARNING: {len(panel_only)} sensory records won't have carcass data")
    # print(f"First 10 panel-only eartags: {list(panel_only)[:10]}")
    print(f"First 10 panel-only eartags: {list(panel_only)}")

if len(carcass_only) > 0:
    print(f"\n⚠️ Note: {len(carcass_only)} carcasses don't have sensory data")
    # print(f"First 10 carcass-only eartags: {list(carcass_only)[:10]}")
    print(f"First 10 carcass-only eartags: {list(carcass_only)}")

# Calculate merge success rate
merge_rate = len(common_eartags) / len(panel_eartags) * 100
print(f"\nMerge success rate: {merge_rate:.1f}% of sensory records can be matched")

## 3. Visualize Data Completeness

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Cuts per carcass
cuts_per_carcass['Number_of_Cuts'].value_counts().sort_index().plot(kind='bar', ax=axes[0])
axes[0].set_title('Distribution of Cuts per Carcass')
axes[0].set_xlabel('Number of Cuts')
axes[0].set_ylabel('Number of Carcasses')
axes[0].axvline(x=3.5, color='r', linestyle='--', alpha=0.5, label='Complete (4 cuts)')
axes[0].legend()

# Plot 2: Panelists per cut
panelists_per_cut['Number_of_Panelists'].value_counts().sort_index().plot(kind='bar', ax=axes[1])
axes[1].set_title('Distribution of Panelists per Cut')
axes[1].set_xlabel('Number of Panelists')
axes[1].set_ylabel('Number of Cuts')
axes[1].axvline(x=5.5, color='r', linestyle='--', alpha=0.5, label='Complete (6 panelists)')
axes[1].legend()

plt.tight_layout()
plt.show()

## 4. Data Quality Summary

Generate a comprehensive summary report

In [ ]:
print("="*60)
print("DATA QUALITY SUMMARY REPORT")
print("="*60)

print("\n1. DATASET SIZES")
print(f"   Sensory panel records: {len(panel_df):,}")
print(f"   Unique carcasses in sensory data: {panel_df['Eartag'].nunique():,}")
print(f"   Unique carcasses in carcass data: {carcass_df['Eartag'].nunique():,}")
print(f"   Carcasses in BOTH datasets: {len(common_eartags):,}")

print("\n2. CUT COMPLETENESS")
complete_carcasses = (cuts_per_carcass['Number_of_Cuts'] == 4).sum()
total_carcasses = len(cuts_per_carcass)
print(f"   Carcasses with all 4 cuts: {complete_carcasses} / {total_carcasses} ({complete_carcasses/total_carcasses*100:.1f}%)")

print("\n3. PANELIST COMPLETENESS")
complete_panels = (panelists_per_cut['Number_of_Panelists'] == 6).sum()
total_cuts = len(panelists_per_cut)
print(f"   Cuts with 6 panelists: {complete_panels} / {total_cuts} ({complete_panels/total_cuts*100:.1f}%)")

print("\n4. EXPECTED MERGED DATASET SIZE")
# Calculate expected rows after merge
mergeable_records = panel_df[panel_df['Eartag'].isin(common_eartags)]
print(f"   Total records that can be merged: {len(mergeable_records):,}")
print(f"   Expected rows (all 4 cuts, 6 panelists): {len(common_eartags) * 4 * 6:,}")
print(f"   Actual mergeable rows: {len(mergeable_records):,}")
if len(mergeable_records) < len(common_eartags) * 4 * 6:
    missing = len(common_eartags) * 4 * 6 - len(mergeable_records)
    print(f"   ⚠️ Missing {missing:,} records ({missing/(len(common_eartags)*4*6)*100:.1f}%)")

print("\n5. RECOMMENDATION")
if complete_carcasses / total_carcasses >= 0.95 and complete_panels / total_cuts >= 0.95:
    print("   ✅ Data quality is EXCELLENT. Proceed with merge.")
elif complete_carcasses / total_carcasses >= 0.80 and complete_panels / total_cuts >= 0.80:
    print("   ⚠️ Data quality is GOOD but some records are incomplete.")
    print("   Consider filtering to complete records only.")
else:
    print("   ❌ Data quality issues detected.")
    print("   Review incomplete records before proceeding.")

print("\n" + "="*60)

## 5. Decision Point: Filter or Keep Incomplete Data?

Based on the summary above, decide whether to:
- Keep ALL data (including incomplete)
- Filter to only complete carcasses (all 4 cuts)
- Filter to only complete panels (6 panelists per cut)

In [ ]:
# Option 1: Keep all data
filtered_panel_df = panel_df.copy()

# Option 2: Filter to complete carcasses only (uncomment to use)
# complete_eartags = cuts_per_carcass[cuts_per_carcass['Number_of_Cuts'] == 4]['Eartag']
# filtered_panel_df = panel_df[panel_df['Eartag'].isin(complete_eartags)].copy()

# Option 3: Filter to complete panels only (uncomment to use)
# complete_cuts = panelists_per_cut[panelists_per_cut['Number_of_Panelists'] == 6]
# complete_cut_ids = complete_cuts.apply(lambda x: (x['Eartag'], x['Primal']), axis=1)
# filtered_panel_df = panel_df[panel_df.apply(lambda x: (x['Eartag'], x['Primal']) in complete_cut_ids.values, axis=1)].copy()

print(f"Original sensory records: {len(panel_df):,}")
print(f"Filtered sensory records: {len(filtered_panel_df):,}")
print(f"Records removed: {len(panel_df) - len(filtered_panel_df):,}")

## 6. Export Data Quality Report

Save the completeness analysis to Excel for review

In [ ]:
with pd.ExcelWriter('data_quality_report.xlsx', engine='openpyxl') as writer:
    # Sheet 1: Cuts per carcass
    cuts_per_carcass.to_excel(writer, sheet_name='Cuts_Per_Carcass', index=False)
    
    # Sheet 2: Panelists per cut
    panelists_per_cut.to_excel(writer, sheet_name='Panelists_Per_Cut', index=False)
    
    # Sheet 3: Incomplete carcasses
    if len(incomplete) > 0:
        incomplete.to_excel(writer, sheet_name='Incomplete_Carcasses', index=False)
    
    # Sheet 4: Incomplete panels
    if len(incomplete_panels) > 0:
        incomplete_panels.to_excel(writer, sheet_name='Incomplete_Panels', index=False)
    
    # Sheet 5: Eartag matching summary
    matching_summary = pd.DataFrame({
        'Category': ['Panel Only', 'Carcass Only', 'Both Datasets'],
        'Count': [len(panel_only), len(carcass_only), len(common_eartags)]
    })
    matching_summary.to_excel(writer, sheet_name='Eartag_Matching', index=False)

print("✅ Data quality report saved to 'data_quality_report.xlsx'")

---
# STOP HERE AND REVIEW

**Before proceeding to the merge:**
1. Review the data quality summary above
2. Open 'data_quality_report.xlsx' to see detailed completeness analysis
3. Decide on filtering strategy (keep all vs. filter incomplete)
4. Check that primal codes match expectations

**Once satisfied, proceed to Part 2 of this notebook for the actual merge.**

---

In [ ]:
# Clean up duplicate names in ther dataset

In [ ]:
path = f'C:'
input_file = f"{path}Measure_DATA_CleanedUp.csv"
eo_a = pd.read_csv(input_file, encoding='utf-8-sig')
eo_a.columns = eo_a.columns.str.replace(r'\ufeff|\ufffe', '', regex=True).str.strip()

In [ ]:
panelist_map = {
    'name_a': 'name_A',
    'name_b': 'name_B',
    'name_c': 'name_C',
    'name_d': 'name_D',
    'name_e': 'name_E',
}
eo_a['Username'] = eo_a['Username'].replace(panelist_map)

In [ ]:
output_file = f"{path}Panel Sensory Data_fixed_2.1.csv"
eo_a.to_csv(output_file, index=False, encoding="utf-8-sig")

In [ ]:
eo_a = eo_a.drop_duplicates(keep="first")
output_file = f"{path}Measure_DATA_CleanedUp_2.0.csv"
eo_a.to_csv(output_file, index=False, encoding="utf-8-sig")

In [ ]:
eo_a = eo_a.drop_duplicates(
    subset=['Eartag', 'Primal', 'Code'], 
    keep='first'
)


In [ ]:
output_file = f"{path}Measure_DATA_CleanedUp_2.2.csv"
eo_a.to_csv(output_file, index=False, encoding="utf-8-sig")

In [ ]:
#  New Merge code

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# ============================================================
# MERGE DATASETS & CREATE TRAIN/TEST SPLITS
# ============================================================

print("="*60)
print("MERGING MEAT QUALITY DATASETS")
print("="*60)

# Assuming you already have:
# measure_df, carcass_df, panel_df loaded

print(f"\nStarting datasets:")
print(f"  Carcass: {carcass_df.shape[0]:,} animals × {carcass_df.shape[1]} columns")
print(f"  Measure: {measure_df.shape[0]:,} steaks × {measure_df.shape[1]} columns")
print(f"  Panel:   {panel_df.shape[0]:,} observations × {panel_df.shape[1]} columns")

# ============================================================
# STEP 1: CREATE STEAK_ID IN PANEL DATA
# ============================================================

print("\n[Step 1/5] Creating steak_id in Panel data...")

# steak_id = Eartag_Primal_ProductCode
panel_df['steak_id'] = (
    panel_df['Eartag'].astype(str) + '_' +
    panel_df['Primal'].astype(str) + '_' +
    panel_df['ProductCode'].astype(str)
)

unique_steaks_panel = panel_df['steak_id'].nunique()
print(f"  ✅ Created {unique_steaks_panel:,} unique steak_ids in Panel")
print(f"  Example: {panel_df['steak_id'].iloc[0]}")

# Start with panel as base
df = panel_df.copy()

# ============================================================
# STEP 2: MERGE WITH MEASURE DATA
# ============================================================

print("\n[Step 2/5] Merging with Measure data...")

# Create steak_id in measure data first
measure_df['steak_id'] = (
    measure_df['Eartag'].astype(str) + '_' +
    measure_df['Primal'].astype(str) + '_' +
    measure_df['Code'].astype(str)
)

print(f"  Created {measure_df['steak_id'].nunique():,} steak_ids in Measure")

# Merge on steak_id (this ensures exact matching)
df = df.merge(
    measure_df,
    on='steak_id',
    how='left',
    suffixes=('', '_measure')
)

# Drop duplicate columns from measure
duplicate_cols = ['Eartag_measure', 'Primal_measure']
duplicate_cols = [c for c in duplicate_cols if c in df.columns]
if duplicate_cols:
    df = df.drop(columns=duplicate_cols)

# Check measure coverage
measure_coverage = df['RAW WEIGHT'].notna().sum() if 'RAW WEIGHT' in df.columns else 0
print(f"  ✅ With measure data: {measure_coverage:,} / {len(df):,} ({measure_coverage/len(df)*100:.1f}%)")

# ============================================================
# STEP 3: MERGE WITH CARCASS DATA
# ============================================================

print("\n[Step 3/5] Merging with Carcass data...")

# Merge on Eartag to get all farm/animal data
df = df.merge(
    carcass_df,
    on='Eartag',
    how='left',
    suffixes=('', '_carcass')
)

print(f"  ✅ Full dataset: {len(df):,} observations × {df.shape[1]} columns")

# ============================================================
# STEP 4: REMOVE ANIMALS WITH MISSING CARCASS DATA
# ============================================================

print("\n[Step 4/5] Removing animals with missing carcass data...")

# Check which animals have missing carcass data
# (Animals in Panel but not in Carcass)
panel_eartags = set(panel_df['Eartag'].unique())
carcass_eartags = set(carcass_df['Eartag'].unique())
missing_carcass_eartags = list(panel_eartags - carcass_eartags)

if len(missing_carcass_eartags) > 0:
    print(f"  Found {len(missing_carcass_eartags)} animals with missing carcass data:")
    print(f"  Eartags: {sorted(missing_carcass_eartags)}")
    
    # Count observations to remove
    to_remove = df[df['Eartag'].isin(missing_carcass_eartags)]
    print(f"  Removing: {len(to_remove):,} observations")
    
    # Remove them
    df = df[~df['Eartag'].isin(missing_carcass_eartags)].copy()
    print(f"  ✅ Cleaned dataset: {len(df):,} observations")
else:
    print(f"  ✅ All animals have carcass data - no removals needed")

# ============================================================
# STEP 5: QUALITY CHECKS
# ============================================================

print("\n" + "="*60)
print("MERGED DATASET SUMMARY")
print("="*60)

print(f"\nDimensions:")
print(f"  Total observations: {len(df):,}")
print(f"  Unique animals (Eartag): {df['Eartag'].nunique()}")
print(f"  Unique steaks (steak_id): {df['steak_id'].nunique()}")
print(f"  Unique panelists: {df['Username'].nunique()}")
print(f"  Total features: {df.shape[1]}")

# Show sample steak_ids
print(f"\nSample steak_ids (Eartag_Primal_Code format):")
print(df['steak_id'].head(10).tolist())

# Check panelists per steak
obs_per_steak = df.groupby('steak_id').size()
print(f"\nPanelists per steak:")
print(f"  Min: {obs_per_steak.min()}")
print(f"  Max: {obs_per_steak.max()}")
print(f"  Mean: {obs_per_steak.mean():.2f}")
print(f"  Median: {obs_per_steak.median():.0f}")

# Verify steak_id format
print(f"\nVerifying steak_id format (Eartag_Primal_Code):")
sample_steak = df.iloc[0]
expected_id = f"{sample_steak['Eartag']}_{sample_steak['Primal']}_{sample_steak['ProductCode']}"
actual_id = sample_steak['steak_id']
print(f"  Expected: {expected_id}")
print(f"  Actual:   {actual_id}")
print(f"  Match: {'✅' if expected_id == actual_id else '❌'}")

# ============================================================
# TRAIN/TEST SPLIT (80/20 BY STEAK_ID)
# ============================================================

print("\n" + "="*60)
print("CREATING TRAIN/TEST SPLIT (80/20)")
print("="*60)

# Get unique steaks
unique_steaks = df['steak_id'].unique()
print(f"\nTotal unique steaks: {len(unique_steaks):,}")

# Split steaks 80/20
train_steaks, test_steaks = train_test_split(
    unique_steaks,
    test_size=0.2,
    random_state=42
)

print(f"\nSteak split:")
print(f"  Train steaks: {len(train_steaks):,} ({len(train_steaks)/len(unique_steaks)*100:.1f}%)")
print(f"  Test steaks:  {len(test_steaks):,} ({len(test_steaks)/len(unique_steaks)*100:.1f}%)")

# Create train and test dataframes
train_df = df[df['steak_id'].isin(train_steaks)].copy()
test_df = df[df['steak_id'].isin(test_steaks)].copy()

print(f"\nObservation split:")
print(f"  Train observations: {len(train_df):,} ({len(train_df)/len(df)*100:.1f}%)")
print(f"  Test observations:  {len(test_df):,} ({len(test_df)/len(df)*100:.1f}%)")

# ============================================================
# VERIFY NO DATA LEAKAGE
# ============================================================

print("\n" + "="*60)
print("DATA LEAKAGE CHECK")
print("="*60)

# Check for overlap
overlap_steaks = set(train_steaks) & set(test_steaks)
print(f"\nSteaks in both train AND test: {len(overlap_steaks)}")

if len(overlap_steaks) == 0:
    print("✅ PERFECT - Zero overlap! No data leakage.")
else:
    print(f"❌ WARNING - {len(overlap_steaks)} steaks appear in BOTH sets!")
    print(f"Overlapping steaks: {list(overlap_steaks)[:10]}")

# Verify no animals appear in both sets
train_animals = set(train_df['Eartag'].unique())
test_animals = set(test_df['Eartag'].unique())
overlap_animals = train_animals & test_animals

print(f"\nAnimals in both train AND test: {len(overlap_animals)}")
if len(overlap_animals) > 0:
    print(f"  ⚠️  Note: {len(overlap_animals)} animals have steaks in BOTH sets")
    print(f"  This is OK - different steaks (primals/codes) from same animal")
else:
    print(f"  ✅ No animal overlap")

# Add split indicator to main dataframe
df['split'] = 'train'
df.loc[df['steak_id'].isin(test_steaks), 'split'] = 'test'

print(f"\nSplit column added to dataframe:")
print(df['split'].value_counts())

# ============================================================
# SUMMARY
# ============================================================

print("\n" + "="*60)
print("✅ MERGE COMPLETE - READY FOR MODELING")
print("="*60)

print(f"""
Available dataframes:
  • df        - Full merged dataset ({len(df):,} obs)
  • train_df  - Training set ({len(train_df):,} obs, {train_df['steak_id'].nunique()} steaks)
  • test_df   - Test set ({len(test_df):,} obs, {test_df['steak_id'].nunique()} steaks)

steak_id format: Eartag_Primal_Code
  Example: 4924_LD_358
  - Eartag: 4924 (animal identifier)
  - Primal: LD (muscle type)
  - Code: 358 (steak number)

Key columns:
  • steak_id  - Unique steak identifier (for grouping)
  • split     - 'train' or 'test' indicator
  • Eartag    - Animal identifier
  • Primal    - Muscle type (LD, SM, GM, FT)
  • Username  - Panelist identifier
  • Tenderness, Juicy, Flavour, Acceptability - Target variables
  
Next steps:
  1. Select features and target
  2. Create CatBoost Pool with group_id=steak_id
  3. Train model
""")

print("="*60)

In [ ]:
output_file = f"{path}New_merged_meat_1.0.csv"
df.to_csv(output_file, index=False, encoding="utf-8-sig")

In [ ]:
df.head(2)